# Weather Rain MLOps: Kaggle launcher

Runs the training sections (3, 5, 6, 7) on a Kaggle GPU. The code comes from the public GitHub repo, the images from a private Kaggle dataset, and every result is written to `/kaggle/working/experiments`.

One-time setup in the notebook editor (right panel):
1. Session options: Accelerator = GPU T4 x2, Internet = on (internet needs a phone-verified Kaggle account).
2. Add Input > Datasets > Your Work: add the weather dataset made from `weather_data_kaggle.zip`.

Every run:
1. Push the latest code to GitHub first. The launcher clones `main` and stops if Section files are missing.
2. Set SECTION, GROUP and KAGGLE_GPU_HOURS_REMAINING (the GPU hours left this week, shown in the right panel or under Settings). The timing step stops before training if the worst case is above 60% of those hours.
3. Save Version > Save & Run All (Commit). The run continues in the background (up to 12 hours) after you close the browser.
4. When the version finishes, open it, go to the Output tab and download `section<N>_results.zip`. Unzip it into the local repo's `experiments/` folder.

Resume: to continue after a stopped or failed version, Add Input > Notebooks > this notebook (the previous version's output), then commit again. Finished runs are copied in and skipped.

No passwords or tokens are used here.

In [ ]:
SECTION = 3
GROUP = "smoke,timing,A,B,C"
KAGGLE_GPU_HOURS_REMAINING = 30
GITHUB_URL = "https://github.com/Flaviaa4/weather_rain_mlops.git"
GIT_REF = "main"
PINNED = {"tensorflow": "2.21.0", "keras": "3.13.2", "tf2onnx": "1.17.0", "onnx": "1.23.2"}
EXPECTED_COUNTS = {"train": 719, "valid": 144, "test": 134}
INPUT_ROOT = "/kaggle/input"
WORK_DIR = "/kaggle/working"
REPO_DIR = "/tmp/weather_rain_mlops"
LOCAL_DATA = "/tmp/data"
EXPERIMENTS_DIR = f"{WORK_DIR}/experiments"
ENV_DIR = f"{EXPERIMENTS_DIR}/kaggle_env"

In [ ]:
import glob, json, os, shutil, subprocess, sys
from importlib import metadata

os.environ.update({"PYTHONUNBUFFERED": "1", "DATA_DIR": LOCAL_DATA, "EXPERIMENTS_DIR": EXPERIMENTS_DIR,
                   "COLAB_UNITS_PER_HOUR": "1", "COLAB_UNITS_REMAINING": str(KAGGLE_GPU_HOURS_REMAINING)})
# one GPU only, so epoch times and results compare with a single Colab T4
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
def count_rows(path):
    with open(path) as handle:
        return sum(1 for _ in handle) - 1


def find_previous_results(root):
    found = glob.glob(f"{root}/**/experiments/results.csv", recursive=True)
    return max(found, key=count_rows, default=None)


previous = find_previous_results(INPUT_ROOT)
if os.path.exists(f"{EXPERIMENTS_DIR}/results.csv"):
    print("results.csv already in /kaggle/working, keeping it")
elif previous:
    shutil.copytree(os.path.dirname(previous), EXPERIMENTS_DIR, dirs_exist_ok=True)
    print(f"Resumed {count_rows(previous)} finished runs from {previous}")
else:
    print("No previous results attached, starting fresh")
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!{sys.executable} --version > {ENV_DIR}/python_version.txt
!{sys.executable} -m pip list > {ENV_DIR}/pip_list_preinstalled.txt 2>/dev/null
print("Preinstalled pip list saved to", f"{ENV_DIR}/pip_list_preinstalled.txt")

In [ ]:
REQUIRED_FILES = ["src/experiments.py", "src/grids.py", "src/model.py", "src/metrics.py", "src/run_report.py",
                  "requirements-trainer.txt"]

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, GITHUB_URL, REPO_DIR], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout.strip()
open(f"{ENV_DIR}/git_commit.txt", "w").write(commit + "\n")
missing = [name for name in REQUIRED_FILES if not os.path.exists(f"{REPO_DIR}/{name}")]
if missing:
    raise SystemExit(f"The GitHub repo is missing {missing}. Commit and push them, then run again.")
print("Cloned", commit)

In [ ]:
def has_splits(path):
    return all(os.path.isdir(f"{path}/{split}") for split in EXPECTED_COUNTS)


def find_dataset_root(root):
    for path, _, _ in os.walk(root):
        if has_splits(path):
            return path
    raise SystemExit(f"No folder with train/, valid/ and test/ under {root}. Add the weather dataset with Add Input.")


def count_images(split_dir):
    return {name: len(os.listdir(f"{split_dir}/{name}")) for name in sorted(os.listdir(split_dir))}


source = find_dataset_root(INPUT_ROOT)
shutil.rmtree(LOCAL_DATA, ignore_errors=True)
shutil.copytree(source, LOCAL_DATA, ignore=shutil.ignore_patterns("quarantine", "uploads", "*.db", "*.zip"))
counts = {split: count_images(f"{LOCAL_DATA}/{split}") for split in EXPECTED_COUNTS}
print("Dataset from", source)
for split, per_class in counts.items():
    print(split, per_class, sum(per_class.values()))
wrong = {split: sum(per_class.values()) for split, per_class in counts.items() if sum(per_class.values()) != EXPECTED_COUNTS[split]}
if wrong:
    raise SystemExit(f"Image counts {wrong} differ from the cleaned dataset {EXPECTED_COUNTS}. Upload the cleaned data, not the original export.")

In [ ]:
def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def pip_install(args, log_name):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *args],
                            cwd=REPO_DIR, capture_output=True, text=True)
    open(f"{ENV_DIR}/{log_name}", "w").write(result.stdout + result.stderr)
    print(result.stdout[-1500:], result.stderr[-1500:])
    if result.returncode != 0:
        raise SystemExit(f"pip install failed, see {ENV_DIR}/{log_name}")


preinstalled_tf = installed_version("tensorflow")
print("Preinstalled TensorFlow:", preinstalled_tf)
cuda_tf = [] if preinstalled_tf == PINNED["tensorflow"] else [f"tensorflow[and-cuda]=={PINNED['tensorflow']}"]
pip_install(["-r", "requirements-trainer.txt", "pytest==9.2.0", *cuda_tf], "pip_install.log")

In [ ]:
PROBE = ("import json, sys, tensorflow as tf, importlib.metadata as m; "
         "print(json.dumps({'python': sys.version.split()[0], 'tensorflow': tf.__version__, "
         "**{name: m.version(name) for name in ['keras', 'tf2onnx', 'onnx', 'onnxruntime', 'numpy']}, "
         "'gpus': [d.name for d in tf.config.list_physical_devices('GPU')]}))")

probe = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit(probe.stderr[-3000:])
versions = json.loads(probe.stdout.strip().splitlines()[-1])
json.dump(versions, open(f"{ENV_DIR}/versions.json", "w"), indent=2)
print(json.dumps(versions, indent=2))
wrong = {name: versions[name] for name in PINNED if versions[name] != PINNED[name]}
if wrong or not versions["gpus"]:
    raise SystemExit(f"Version mismatch {wrong} or no GPU {versions['gpus']}. Set Accelerator to GPU T4 x2 and run again.")
check = subprocess.run([sys.executable, "-m", "src.check_env", "--profile", "trainer"], cwd=REPO_DIR, capture_output=True, text=True)
open(f"{ENV_DIR}/check_env_kaggle.txt", "w").write(check.stdout + check.stderr)
print(check.stdout[-3000:])

In [ ]:
def stream(command, log_path):
    with open(log_path, "a") as log, subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                                       stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
            log.write(line)
    return process.returncode


def zip_results():
    archive = shutil.make_archive(f"{WORK_DIR}/section{SECTION}_results", "zip", EXPERIMENTS_DIR)
    print(f"Download {os.path.basename(archive)} ({os.path.getsize(archive) / 1e6:.0f} MB) from the Output tab")


LOG_TAG = GROUP.replace(",", "_")
try:
    tests_log = f"{EXPERIMENTS_DIR}/kaggle_section{SECTION}_tests.log"
    if "smoke" in GROUP and stream([sys.executable, "-m", "pytest", "-q", "tests"], tests_log) != 0:
        raise SystemExit("Tests failed, see the output above. No experiment was started.")
    command = [sys.executable, "-m", "src.experiments", "--section", str(SECTION), "--group", GROUP]
    code = stream(command, f"{EXPERIMENTS_DIR}/kaggle_section{SECTION}_{LOG_TAG}.log")
    if code != 0:
        raise SystemExit(f"src.experiments exited with code {code}, see the output above. Finished runs are kept.")
finally:
    zip_results()